CELL 1 — Clone DBIQA

In [ ]:
!git clone https://github.com/Buka-Xing/Dual-Branch-Image-Quality-Assessment

%cd Dual-Branch-Image-Quality-Assessment

Cloning into 'Dual-Branch-Image-Quality-Assessment'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 226 (delta 65), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (226/226), 29.69 MiB | 10.58 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/Dual-Branch-Image-Quality-Assessment


CELL 2 — Install Requirements

In [ ]:
!pip install scipy
!pip install pandas
!pip install openpyxl

CELL 3 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!cp /content/drive/MyDrive/DBIQA_Models/DBIQA_*_ASPP.py /content/Dual-Branch-Image-Quality-Assessment/

CELL 4 — Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import glob

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

from sklearn.model_selection import train_test_split

from scipy.stats import pearsonr
from scipy.stats import spearmanr
torch.backends.cudnn.benchmark = True
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


CELL 5 — Dataset Paths

In [ ]:
IMG_DIR = "/content/drive/MyDrive/image"

REF_IMAGE = "3.png"

DISTORTED_IMAGES = {
    "Gaussian Blur": "image3_gb3.png",
    "Gaussian Noise": "image3_gn3.png",
    "Low Light": "image3_ll3.png",
    "Motion Blur": "image3_mb3.png"
}

print("Images found:")

for filename in os.listdir(IMG_DIR):
    print(filename)

Images found:
3.png
image3_gn3.png
image3_gb3.png
image3_ll3.png
image3_mb3.png


In [ ]:
REF_DIR = "/content/drive/MyDrive/image"

DIST_DIR = "/content/drive/MyDrive/image/dist"

print("Reference Images :", len(os.listdir(REF_DIR)))
print("Blurred Images   :", len(os.listdir(DIST_DIR)))

Reference Images : 5


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/image/dist'

CELL 6 — Model Folder

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/DBIQA_RESULTS/CSIQ"

print("\nModels Found")

model_files = sorted(glob.glob(os.path.join(MODEL_DIR,"*.pth")))

for f in model_files:
    print(os.path.basename(f))


Models Found
best_dbiqa_efficientnet.pth
best_dbiqa_efficientnet_aspp.pth
best_dbiqa_mobilenetv2.pth
best_dbiqa_mobilenetv2_aspp.pth
best_dbiqa_resnet50.pth
best_dbiqa_resnet50_aspp.pth
best_dbiqa_squeeze.pth
best_dbiqa_squeeze_aspp.pth
best_dbiqa_vgg19.pth
best_dbiqa_vgg19_aspp.pth


CELL 7 — Disable ImageNet Pretrained Weights

In [ ]:
files = [

    "DBIQA_VGG.py",
    "DBIQA_ResNet.py",
    "DBIQA_Squeeze.py",
    "DBIQA_mobileV2.py",
    "DBIQA_EfficientNet.py"

]

for file_path in files:

    with open(file_path,"r") as f:

        text = f.read()

    text = text.replace(
        "pretrained=True",
        "pretrained=False"
    )

    with open(file_path,"w") as f:

        f.write(text)

print("Done.")

Done.


CELL 8 — Import Every Backbone

In [ ]:
from DBIQA_VGG import DBIQA as DBIQA_VGG
from DBIQA_VGG import VGG

from DBIQA_VGG_ASPP import DBIQA as DBIQA_VGG_ASPP
from DBIQA_VGG_ASPP import VGG as VGG_ASPP

from DBIQA_ResNet import DBIQA as DBIQA_RESNET
from DBIQA_ResNet import ResNet50

from DBIQA_ResNet_ASPP import DBIQA as DBIQA_RESNET_ASPP
from DBIQA_ResNet_ASPP import ResNet50 as ResNet50_ASPP

from DBIQA_Squeeze import DBIQA as DBIQA_SQUEEZE
from DBIQA_Squeeze import Squeeze

from DBIQA_Squeeze_ASPP import DBIQA as DBIQA_SQUEEZE_ASPP
from DBIQA_Squeeze_ASPP import Squeeze as Squeeze_ASPP

from DBIQA_mobileV2 import DBIQA as DBIQA_MOBILE
from DBIQA_mobileV2 import mobileV2

from DBIQA_mobileV2_ASPP import DBIQA as DBIQA_MOBILE_ASPP
from DBIQA_mobileV2_ASPP import mobileV2 as mobileV2_ASPP

from DBIQA_EfficientNet import DBIQA as DBIQA_EFF
from DBIQA_EfficientNet import efficientB0

from DBIQA_EfficientNet_ASPP import DBIQA as DBIQA_EFF_ASPP
from DBIQA_EfficientNet_ASPP import efficientB0 as efficientB0_ASPP

print("All models imported successfully.")

All models imported successfully.


CELL 9 — Image Preprocessing

In [ ]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor()

])

def load_pair(ref_path,dist_path):

    ref = Image.open(ref_path).convert("RGB")

    dist = Image.open(dist_path).convert("RGB")

    ref = transform(ref).unsqueeze(0).to(device)

    dist = transform(dist).unsqueeze(0).to(device)

    return ref,dist

print("Image loader ready.")

Image loader ready.


CELL 10 — Model Loader

In [ ]:
BACKBONES = [

    "vgg19",
    "vgg19_aspp",

    "resnet50",
    "resnet50_aspp",

    "squeeze",
    "squeeze_aspp",

    "mobilenetv2",
    "mobilenetv2_aspp",

    "efficientnet",
    "efficientnet_aspp"

]

print(BACKBONES)

def load_model(backbone):

    if backbone == "vgg19":

        from DBIQA_VGG import DBIQA, VGG

        feature_net = VGG(
            pretrained=False,
            requires_grad=False
        )

    elif backbone == "vgg19_aspp":

        from DBIQA_VGG_ASPP import DBIQA, VGG

        feature_net = VGG(
            pretrained=False,
            requires_grad=False
        )

    elif backbone == "resnet50":

        from DBIQA_ResNet import DBIQA, ResNet50

        feature_net = ResNet50(
            requires_grad=False
        )

    elif backbone == "resnet50_aspp":

        from DBIQA_ResNet_ASPP import DBIQA, ResNet50

        feature_net = ResNet50(
            requires_grad=False
        )

    elif backbone == "squeeze":

        from DBIQA_Squeeze import DBIQA, Squeeze

        feature_net = Squeeze(
            requires_grad=False
        )

    elif backbone == "squeeze_aspp":

        from DBIQA_Squeeze_ASPP import DBIQA, Squeeze

        feature_net = Squeeze(
            requires_grad=False
        )

    elif backbone == "mobilenetv2":

        from DBIQA_mobileV2 import DBIQA, mobileV2

        feature_net = mobileV2(
            requires_grad=False
        )

    elif backbone == "mobilenetv2_aspp":

        from DBIQA_mobileV2_ASPP import DBIQA, mobileV2

        feature_net = mobileV2(
            requires_grad=False
        )

    elif backbone == "efficientnet":

        from DBIQA_EfficientNet import DBIQA, efficientB0

        feature_net = efficientB0(
            requires_grad=False
        )

    elif backbone == "efficientnet_aspp":

        from DBIQA_EfficientNet_ASPP import DBIQA, efficientB0

        feature_net = efficientB0(
            requires_grad=False
        )

    else:
        raise ValueError(backbone)

    feature_net = feature_net.to(device)

    model = DBIQA().to(device)

    ckpt = os.path.join(
        MODEL_DIR,
        f"best_dbiqa_{backbone}.pth"
    )

    checkpoint = torch.load(
        ckpt,
        map_location=device,
        weights_only=False
    )

    feature_net.load_state_dict(
        checkpoint["feature_net"]
    )

    model.load_state_dict(
        checkpoint["dbiqa"]
    )

    feature_net.eval()
    model.eval()

    print(f"Loaded {backbone}")

    return feature_net, model

['vgg19', 'vgg19_aspp', 'resnet50', 'resnet50_aspp', 'squeeze', 'squeeze_aspp', 'mobilenetv2', 'mobilenetv2_aspp', 'efficientnet', 'efficientnet_aspp']


CELL 11 — Inference Function

In [ ]:
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


def predict_score(feature_net,
                  model,
                  ref_path,
                  dist_path):

    ref = Image.open(ref_path).convert("RGB")
    dist = Image.open(dist_path).convert("RGB")

    ref = transform(ref).unsqueeze(0).to(device)
    dist = transform(dist).unsqueeze(0).to(device)

    with torch.no_grad():

        ref_feat = feature_net(ref)
        dist_feat = feature_net(dist)

        score = model(
            ref_feat,
            dist_feat,
            as_loss=False
        )

    return float(score.item())

CELL 12 — Run All Models on All Image Pairs

In [ ]:
loaded_models = {}

for backbone in BACKBONES:
    loaded_models[backbone] = load_model(backbone)

IMG_DIR = "/content/drive/MyDrive/image"

REF_IMAGE = "3.png"

DISTORTED_IMAGES = {
    "Gaussian Blur": "image3_gb3.png",
    "Gaussian Noise": "image3_gn3.png",
    "Low Light": "image3_ll3.png",
    "Motion Blur": "image3_mb3.png"
}

ref_path = os.path.join(IMG_DIR, REF_IMAGE)

results = []

for distortion, dist_name in DISTORTED_IMAGES.items():

    dist_path = os.path.join(IMG_DIR, dist_name)

    print("=" * 70)
    print("Distortion :", distortion)
    print("Image      :", dist_name)
    print("=" * 70)

    row = {
        "Reference Image": REF_IMAGE,
        "Distorted Image": dist_name,
        "Distortion Type": distortion
    }

    for backbone in BACKBONES:

        feature_net, model = loaded_models[backbone]

        score = predict_score(
            feature_net,
            model,
            ref_path,
            dist_path
        )

        row[backbone] = round(score, 4)

        print(f"{backbone:20s}: {score:.4f}")

    results.append(row)

comparison = pd.DataFrame(results)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("\nDBIQA-ASPP Qualitative Results\n")

display(comparison)

SAVE_DIR = "/content/drive/MyDrive/DBIQA_RESULTS"
os.makedirs(SAVE_DIR, exist_ok=True)

csv_path = os.path.join(
    SAVE_DIR,
    "Roadside_Qualitative_Results.csv"
)

xlsx_path = os.path.join(
    SAVE_DIR,
    "Roadside_Qualitative_Results.xlsx"
)

comparison.to_csv(csv_path, index=False)
comparison.to_excel(xlsx_path, index=False)

print("CSV saved to:")
print(csv_path)

print("\nExcel saved to:")
print(xlsx_path)